Installation

In [ ]:
!pip install -U datasets transformers accelerate trl sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 38.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 842.4/842.4 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 14.4 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.12.1
    Uninstalling transformers-5.12.1:
      Successfully uninstalled transformers-5.12.1


In [ ]:
import os
import gc
import json
import re
import torch

from datasets import load_dataset, concatenate_datasets, DatasetDict
from transformers import AutoTokenizer, AutoModelForCausalLM

from trl import SFTTrainer, SFTConfig

In [ ]:
print("CUDA verfügbar:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA Version:", torch.version.cuda)
else:
    print("Keine GPU im aktuellen Kernel.")

CUDA verfügbar: True
GPU: Tesla T4
CUDA Version: 12.8


In [ ]:
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

model_name = "LSX-UniWue/LLaMmlein_120M"

tokenizer = AutoTokenizer.from_pretrained(model_name)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

print("Model device:", next(model.parameters()).device)

Imports

In [ ]:
local_files = {
    "detox_hate_speech": "/content/detox-hate_speech.jsonl",
    "detox_file_2": "/content/detox-sentiment.jsonl",
    "detox_file_3": "/content/detox-toxic.jsonl",
}

public_files = {
    "germeval_engaging": "https://raw.githubusercontent.com/UKPLab/lou-gender-fair-reformulations/main/lou/germeval-engaging.jsonl",
    "germeval_factclaiming": "https://raw.githubusercontent.com/UKPLab/lou-gender-fair-reformulations/main/lou/germeval-factclaiming.jsonl",
    "germeval_toxic": "https://raw.githubusercontent.com/UKPLab/lou-gender-fair-reformulations/main/lou/germeval-toxic.jsonl",
    "x_stance_de": "https://raw.githubusercontent.com/UKPLab/lou-gender-fair-reformulations/main/lou/x-stance-de.jsonl",
}

In [ ]:
def load_and_prepare_file(path, dataset_name):
    ds = load_dataset("json", data_files=path, split="train")

    def make_pairs(example):
        original = example["original"].strip()
        neutral = example["Neutral"].strip()

        return {
            "source": original,
            "target": neutral,
            "dataset_name": dataset_name,
        }

    ds = ds.map(make_pairs)

    ds = ds.remove_columns(
        [col for col in ds.column_names if col not in ["source", "target", "dataset_name"]]
    )

    ds = ds.filter(
        lambda x: (
            x["source"] is not None
            and x["target"] is not None
            and len(x["source"].strip()) > 0
            and len(x["target"].strip()) > 0
            and x["source"].strip() != x["target"].strip()
        )
    )

    return ds

In [ ]:
all_datasets = []

for name, path in local_files.items():
    print(f"Lade lokale Datei: {name}")
    all_datasets.append(load_and_prepare_file(path, name))

for name, path in public_files.items():
    print(f"Lade öffentliche Lou-Datei: {name}")
    all_datasets.append(load_and_prepare_file(path, name))

full_dataset = concatenate_datasets(all_datasets).shuffle(seed=42)

print(full_dataset)
print(full_dataset.column_names)
print(full_dataset[0])
print("Gesamtanzahl:", len(full_dataset))

In [ ]:
def to_prompt_completion(example):
    prompt = (
        "### Aufgabe:\n"
        "Formuliere den folgenden deutschen Satz genderneutral um. "
        "Verändere die Bedeutung nicht. Gib nur die umformulierte Version aus.\n\n"
        "### Eingabe:\n"
        f"{example['source']}\n\n"
        "### Ausgabe:\n"
    )

    completion = example["target"]

    return {
        "text": prompt + completion
    }

llammlein_ds = full_dataset.map(to_prompt_completion)

llammlein_ds = llammlein_ds.remove_columns(
    [col for col in llammlein_ds.column_names if col != "text"]
)

print(llammlein_ds[5]["text"])
print("Anzahl Trainingsbeispiele:", len(llammlein_ds))

Train/Validation/Test Split

In [ ]:
train_test = llammlein_ds.train_test_split(test_size=0.2, seed=42)
valid_test = train_test["test"].train_test_split(test_size=0.5, seed=42)

dataset = DatasetDict({
    "train": train_test["train"],
    "validation": valid_test["train"],
    "test": valid_test["test"],
})

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 1085
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 136
    })
    test: Dataset({
        features: ['text'],
        num_rows: 136
    })
})


In [ ]:
sft_args = SFTConfig(
    output_dir="./GeReT",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-5,
    num_train_epochs=5,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    report_to="none",
    fp16=torch.cuda.is_available(),
    dataset_text_field="text",
    max_length=512,
    packing=False,
)

trainer = SFTTrainer(
    model=model,
    args=sft_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    processing_class=tokenizer,
)

trainer.train()

In [ ]:
final_model_path = "./GeReT/"

trainer.save_model(final_model_path)
tokenizer.save_pretrained(final_model_path)

model = trainer.model
model.eval()

print("Gespeichert unter:", final_model_path)

In [ ]:
gender_dict_path = "/content/gender_dict_geschicktgendern.json"

with open(gender_dict_path, "r", encoding="utf-8") as f:
    gender_dict = json.load(f)

print("Einträge im gender_dict:", len(gender_dict))
print("Zuschauer →", gender_dict.get("Zuschauer"))

Einträge im gender_dict: 1707
Zuschauer → ['Person aus dem Publikum', 'Publikum', 'Zuschauende', 'Schaulustige']


In [ ]:
def parse_marked_input(marked_text):
    marked_terms = re.findall(r"\[\[(.*?)\]\]", marked_text)
    clean_text = re.sub(r"\[\[(.*?)\]\]", r"\1", marked_text)
    return clean_text, marked_terms


def candidate_keys(term):
    """
    Erzeugt mögliche Dictionary-Schlüssel für flektierte Begriffe.
    Beispiel: Wählern -> Kandidat
    """
    term = re.sub(r"^[^\wÄÖÜäöüß]+|[^\wÄÖÜäöüß]+$", "", term)

    candidates = []

    def add(x):
        if x and x not in candidates:
            candidates.append(x)

    add(term)

    if term:
        add(term[0].upper() + term[1:])
        add(term[0].lower() + term[1:])

    base_forms = list(candidates)

    for t in base_forms:
        if t.endswith("n"):
            add(t[:-1])

        if t.endswith("en"):
            add(t[:-2])

        if t.endswith("s"):
            add(t[:-1])

        if t.endswith("es"):
            add(t[:-2])

    return candidates


def find_gender_entry(term, gender_dict):
    for candidate in candidate_keys(term):
        if candidate in gender_dict:
            return candidate, gender_dict[candidate]

    return None, None


def build_gender_hints(marked_terms, gender_dict, max_alternatives=6):
    hint_lines = []

    for term in marked_terms:
        key, alternatives = find_gender_entry(term, gender_dict)

        if alternatives:
            selected = alternatives[:max_alternatives]
            hint_lines.append(
                f"- {term} → Dictionary-Eintrag: {key}; mögliche Alternativen: {', '.join(selected)}"
            )
        else:
            hint_lines.append(
                f"- {term} → keine Wörterbuchalternative gefunden; bitte kontextsensitiv genderneutral umformulieren"
            )

    if not hint_lines:
        return "Keine markierten Begriffe gefunden."

    return "\n".join(hint_lines)